In [ ]:
# Switch path to root of project
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"
current_dir = "/home/xxx/research/MedAlign"
# src_path = os.path.join(current_dir, 'src')
os.chdir(current_dir)

# from open_clip import create_model_and_transforms, get_mean_std
from open_clip import create_model_and_transforms, get_mean_std, HFTokenizer
from PIL import Image
import torch
from urllib.request import urlopen

# Define main parameters
model = 'ViT-L-14-336-quickgelu' # available pretrained weights ['ViT-L-14-336-quickgelu', 'ViT-B-16-quickgelu']
pretrained = "/data/xxx/CLIP/unimed_clip_vit_l14_base_text_encoder.pt" # Path to pretrained weights
text_encoder_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract" # available pretrained weights ["microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract", "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract"]
mean, std = get_mean_std()
device='cuda'

In [ ]:
model_med, _, preprocess = create_model_and_transforms(
    model,
    pretrained,
    precision='amp',
    device=device,
    force_quick_gelu=True,
    pretrained_image=False,
    mean=mean, std=std,
    inmem=True,
    text_encoder_name=text_encoder_name,
)
tokenizer = HFTokenizer(
    text_encoder_name,
    context_length=256,
    **{},
)

In [3]:
model_visual_encoder = model_med.visual.cuda()
model_text_encoder = model_med.text_encoder.cuda()

In [ ]:
# load abdomen vqa data
import json
vqa_train_file = "/data/xxx/hallucination/Slake/data/training_masks.json"
with open(vqa_train_file, "r") as f:
    vqa_data = json.load(f)
# Check the loaded data
print(f"Loaded {len(vqa_data)} vqa data")
# filter only english data
vqa_data_train = vqa_data
print(f"Filtered {len(vqa_data_train)} vqa data")

Loaded 4919 vqa data
Filtered 4919 vqa data


In [ ]:
# sample images
import os
import random
from pathlib import Path

from sklearn.manifold import TSNE
import numpy as np

import matplotlib.pyplot as plt

# Step 1: Locate all `source.jpg` files in subfolders
root_dir = Path("/data/xxx/hallucination/Slake/imgs")
# image_paths = list(root_dir.rglob("source.jpg"))
image_paths = list(set([i['image'] for i in vqa_data_train]))

# Step 2: Randomly sample 100 images
sampled_paths = image_paths

print(f"Found {len(image_paths)} training images, sampled {len(sampled_paths)}")

Found 450 training images, sampled 450


In [6]:
full_sampled_paths = []
for path in sampled_paths:
    full_path = os.path.join(root_dir, path)
    full_sampled_paths.append(full_path)

In [7]:
import torch.nn.functional as F
from tqdm import tqdm

In [8]:

attn_map_dict = {}
start_index, sample_number = 0, 450
ind = 0
cos_matrix_dict = {}
for img_path in tqdm(full_sampled_paths[start_index:start_index+sample_number]):  # label: organ/lesion/other
    # image = Image.open(img_path).convert("RGB")
    # image_size = image.size[::-1]  # (H, W)
    # print(f"Processing {img_path}")
    relative_path = sampled_paths[ind]
    ind += 1
    inputs = preprocess(Image.open(img_path)).to("cuda").unsqueeze(0)

    # Preprocess and get patch tokens
    # inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        vision_output = model_visual_encoder.forward_intermediates(inputs)
        # print(vision_output['image_intermediates'][0].size())
        # patch_tokens = vision_output['image_intermediates'][-1][:, 1:, :].squeeze(0)  # [576, D]
        attention_map = vision_output['attn_weights']
    layer_idx = 23
    attn_map = attention_map[layer_idx].squeeze(0)  # [577, 577]

    # Extract [CLS] attention (row 0)
    cls_attn = attn_map[0]  # Shape [577]

    # Drop the [CLS] token itself (attention to itself)
    cls_to_patches = cls_attn[1:]  # Shape [576]

    # Normalize attention for visualization
    cls_to_patches = cls_to_patches / cls_to_patches.max()

    attn_map_dict[relative_path] = cls_to_patches.cpu().numpy()
    


100%|██████████| 450/450 [00:19<00:00, 22.80it/s]


In [ ]:
# saving use pickle
import pickle
with open("/data/xxx/hallucination/Slake/steering/steer_data/attn_map_dict.pkl", "wb") as f:
    pickle.dump(attn_map_dict, f)

In [10]:
len(attention_map)

24

In [11]:
attention_map[0].shape

torch.Size([1, 577, 577])